# Fractal-Stochastic Hybrid Prediction Engine v1.3

### Combining Hurst Explonent Regime Detection with XGBoost Directional CLassification

### Target: Predict the direction of the S&P 500 tomorrow. 

## Mathematical Adjustments

### 1.a Fractional Differencing

##### Rationale: Fractional Differencing is an improvement upon my prior integer differencing as it maintains some memory along with the stationarity. 

### Integer: 

$$ y_t = x_t - x_{t-1} = (1 - L) x_t $$

### Fractional: 

$$ y_t = (1 - L)^d x_t
= x_t - d x_{t-1} + \frac{d(d-1)}{2} x_{t-2} - \frac{d(d-1)(d-2)}{6} x_{t-3} + \cdots $$

### 1.b Hurst Exponent Regime Classification

### Rationale: The Hurst exponent H classifies market regimes:
$$ H > 0.5: Persistent $$
$$ H = 0.5: Random Walk $$
$$ H < 0.5: Mean Reverting $$

In [4]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from hurst import compute_Hc
from xgboost import XGBClassifier
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score
from sklearn.calibration import CalibratedClassifierCV
import shap
import warnings
from datetime import datetime
from statsmodels.tsa.stattools import adfuller

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
SEED = 42
np.random.seed(SEED)



In [5]:
def frac_diff_ffd(series, d, thres=1e-5):
    #Const width window fractional differentiation to preserve memory but ensure stationarity
    #compute weights
    w, k = [1.], 1
    while True: 
        w_ = -w[-1] * (d - k + 1) / k
        if abs(w_) < thres: 
            break
        w.append(w_)
        k += 1
    weights = np.array(w[::-1])

    #apply weights to series
    res = np.convolve(series.values.flatten(), weights, mode='valid')
    return pd.Series(res, index=series.index[len(weights)-1:])

In [22]:
#technical indicators
def compute_rsi(returns, span=14):
    gain = returns.clip(lower=0).ewm(span=span, adjust=False).mean()
    loss = (-returns.clip(upper=0)).ewm(span=span, adjust=False).mean()
    rs = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

def compute_macd(close, fast=12, slow=26, signal=9):
    ema_fast = close.ewm(span=fast, adjust=False).mean()
    ema_slow = close.ewm(span=slow, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    return macd_line - signal_line

def compute_atr(high, low, close, window=14):
    tr = pd.concat([
        high - low,
        (high - close.shift()).abs(),
        (low - close.shift()).abs()
    ], axis=1).max(axis=1)
    return tr.rolling(window).mean()

def max_drawdown(returns):
    equity = (1 + returns).cumprod()
    rolling_max = equity.cummax()
    drawdown = (equity - rolling_max) / rolling_max
    return drawdown.min()

def calculate_sharpe(rets):
    return (rets.mean() / rets.std()) * np.sqrt(252) if rets.std() != 0 else 0

def calculate_calmar(returns, freq=252):
    ann_return = (1 + returns.mean()) ** freq - 1
    mdd = abs(max_drawdown(returns))
    return ann_return / mdd if mdd > 0 else np.nan

### Cell 2: Data Acquisition & Stationarity

In [10]:
data = yf.download('^GSPC', start='2000-01-01', end=datetime.now().strftime('%Y-%m-%d'))
data = data.dropna()

data['FracDiff_Close'] = frac_diff_ffd(data['Close'], d=0.4)

#fractal features
#rolling hurst 50d and 150d to detect transitions
h_short, h_long = [], []
for i in range(len(data)):
    if i < 150:
        h_short.append(np.nan); h_long.append(np.nan)
        continue
    h_s = compute_Hc(data['Close'].iloc[i-100:i], kind='price', simplified=True)[0]
    h_l = compute_Hc(data['Close'].iloc[i-150:i], kind='price', simplified=True)[0]
    h_short.append(h_s); h_long.append(h_l)

data['H_short'] = h_short 
data['H_long'] = h_long
data['H_delta'] = data['H_short'] - data['H_long']

# Momentum & Stochastic Features
data['Returns'] = data['Close'].pct_change()
data['RSI'] = compute_rsi(data['Returns'])
data['MACD_hist'] = compute_macd(data['Close'])
data['Momentum5'] = data['Close'].pct_change(5)
data['Trend'] = (data['Close'].rolling(10).mean() > data['Close'].rolling(50).mean()).astype(int)
data['Volatility'] = data['Returns'].rolling(20).std()

# Volatility & Risk Features
data['ATR_norm'] = compute_atr(data['High'].squeeze(), data['Low'].squeeze(), data['Close'].squeeze()) / data['Close'].squeeze()
bb_mid = data['Close'].rolling(20).mean()
bb_std = data['Close'].rolling(20).std()
data['BB_width'] = (2 * bb_std) / bb_mid

# Interaction Features
data['Hurst_RSI_x'] = data['H_short'] * data['RSI']
data['Hurst_Vol_x'] = data['H_short'] * data['Volatility']
data['Hurst_Mom_x'] = data['H_short'] * data['Momentum5']

[*********************100%***********************]  1 of 1 completed


### training

In [ ]:
FEATURES = [
    'FracDiff_Close', 'H_short', 'H_long', 'H_delta','RSI', 'MACD_hist', 'Momentum5', 'Trend',
    'Volatility', 'ATR_norm', 'BB_width','Hurst_RSI_x', 'Hurst_Vol_x', 'Hurst_Mom_x'
]

data['Target'] = (data['Returns'].shift(-1) > 0).astype(int)
data[FEATURES] = data[FEATURES].shift(1)
data = data.dropna()
adf_result = adfuller(data['FracDiff_Close'])
print(f"ADF Statistic: {adf_result[0]:.4f}, p-value: {adf_result[1]:.4f}")


y = data['Target']
X = data[FEATURES]

split_idx  = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

scaler = StandardScaler()
X_train_sc = pd.DataFrame(scaler.fit_transform(X_train), columns=FEATURES, index=X_train.index)
X_test_sc  = pd.DataFrame(scaler.transform(X_test), columns=FEATURES, index=X_test.index)

tscv = TimeSeriesSplit(n_splits=5, test_size=200, gap=150)

xgb = XGBClassifier(eval_metric='logloss', random_state=SEED)
param_grid = {
    'max_depth': [3, 5],
    'learning_rate': [0.01, 0.05],
    'n_estimators': [100, 200],
    'subsample': [0.8],
    'colsample_bytree': [0.8]
}
grid_search = GridSearchCV(estimator=xgb, param_grid=param_grid, cv=tscv, scoring='roc_auc', n_jobs=-1)
grid_search.fit(X_train_sc, y_train)

best_model = grid_search.best_estimator_


ADF Statistic: -6.4841, p-value: 0.0000

--- PERFORMANCE METRICS ---
Test Accuracy: 0.5362
Test ROC-AUC:  0.5321

Best Parameters found: {'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.8}

Classification Report:
               precision    recall  f1-score   support

           0       0.58      0.01      0.03       476
           1       0.54      0.99      0.70       546

    accuracy                           0.54      1022
   macro avg       0.56      0.50      0.36      1022
weighted avg       0.56      0.54      0.38      1022



### evaluation

In [ ]:
y_pred = best_model.predict(X_test_sc)
test_returns = data['Returns'].loc[X_test.index]
strategy_returns = y_pred * test_returns

print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Test ROC-AUC:  {roc_auc_score(y_test, best_model.predict_proba(X_test_sc)[:, 1]):.4f}")
print("-" * 30)
print(f"Strategy Sharpe Ratio:  {calculate_sharpe(strategy_returns):.4f}")
print(f"Buy & Hold Sharpe Ratio: {calculate_sharpe(test_returns):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Test Accuracy: 0.5362
Test ROC-AUC:  0.5321
------------------------------
Strategy Sharpe Ratio:  0.8066
Buy & Hold Sharpe Ratio: 0.7181

Classification Report:
               precision    recall  f1-score   support

           0       0.58      0.01      0.03       476
           1       0.54      0.99      0.70       546

    accuracy                           0.54      1022
   macro avg       0.56      0.50      0.36      1022
weighted avg       0.56      0.54      0.38      1022

